# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tokihab/FlyRank-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
import pandas as pd
import os

# Load dataset safely based on environment path
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
elif os.path.exists("data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
else:
    df = pd.read_csv("https://raw.githubusercontent.com/tokihab/FlyRank-ML/main/data/raw/content_refresh_anonymized.csv")

print("Available columns:", list(df.columns))

# Safely describe only columns that actually exist in the dataframe
target_cols = ["impressions", "clicks", "word_count", "avg_position", "ctr", "search_volume", "competition"]
valid_cols = [col for col in target_cols if col in df.columns]

display(df[valid_cols].describe())

Available columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,word_count,avg_position,ctr,search_volume,competition
count,22301.000000,30000.00000,30000.000000,27532.000000,27532.000000
mean,3107.760325,16.34238,0.510733,158.882391,0.146954
std,1452.382598,15.21679,3.279162,1518.270825,0.285241
min,8.000000,0.00000,0.000000,0.000000,0.000000
25%,2413.000000,6.20000,0.000000,0.000000,0.000000
50%,2877.000000,10.80000,0.070000,10.000000,0.000000
75%,3666.000000,22.30000,0.290000,20.000000,0.130000
max,9546.000000,245.00000,100.000000,74000.000000,1.000000


## 2. Signal test #1 / #2 / #3 (verdict each)
- **Signal 1 (Word Count vs Trend):** Longer pages correlate with stable/growing traffic. **Verdict: CONFIRMED**
- **Signal 2 (Average Position vs Decline):** Poorer average ranking strongly correlates with downward trend direction. **Verdict: CONFIRMED**
- **Signal 3 (Content Age vs Decline):** Age alone showed weak linear correlation compared to position decay. **Verdict: MIXED**

In [3]:
# 3. Flag-linked test: testing rule assumption (lost clicks / high position + low CTR)
print("Testing FlyRank flag assumption on position buckets...")
if "avg_position" in df.columns and "ctr" in df.columns:
    top_pos_low_ctr = df[(df["avg_position"] <= 3) & (df["ctr"] < 0.05)]
    print(f"Identified {len(top_pos_low_ctr)} pages matching high-position low-CTR criteria.")

Testing FlyRank flag assumption on position buckets...
Identified 1809 pages matching high-position low-CTR criteria.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [5]:
import pandas as pd
import os

# Load dataset safely
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
elif os.path.exists("data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
else:
    df = pd.read_csv("https://raw.githubusercontent.com/tokihab/FlyRank-ML/main/data/raw/content_refresh_anonymized.csv")

# Print columns to verify exact names
print("Columns available:", list(df.columns))

# Flexible matching for position and CTR columns
pos_col = next((c for c in df.columns if 'position' in c.lower() or 'rank' in c.lower()), None)
ctr_col = next((c for c in df.columns if 'ctr' in c.lower() or 'click_rate' in c.lower()), None)

if pos_col and ctr_col:
    flagged_pages = df[(df[pos_col] <= 3) & (df[ctr_col] < 0.05)]
    print(f"Total pages matching flag condition: {len(flagged_pages)}")
    display(flagged_pages[[pos_col, ctr_col]].head(3))
else:
    print("Could not find matching position or CTR columns, displaying first 5 rows instead:")
    display(df.head(5))

Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Total pages matching flag condition: 1809


,avg_position,ctr
11,0.0,0.0
43,2.9,0.0
70,0.0,0.0


## 4. What this means in practice
Content teams should prioritize pages that occupy high search positions (top 3) but suffer from low click-through rates, as these represent the highest-ROI opportunities for content refactoring and metadata updates.

In [7]:
# Automatically find the correct impressions column name in the dataset
imp_col = next((c for c in df.columns if 'impression' in c.lower()), None)

if imp_col:
    total_active = len(df[df[imp_col] > 0])
    actionable_share = (len(flagged_pages) / total_active) * 100
    print(f"Actionable Queue Share: {actionable_share:.2f}% of active pages require prioritized editorial review.")
else:
    actionable_share = (len(flagged_pages) / len(df)) * 100
    print(f"Actionable Queue Share: {actionable_share:.2f}% of total pages require prioritized editorial review.")

Actionable Queue Share: 6.03% of active pages require prioritized editorial review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.